# 🚦 Traffic Demand Prediction — Full ML Pipeline
**Task:** Predict `demand` (0–1 float) for each geohash-day-timestamp combo  
**Metric:** `max(0, 100 × R² score)`  
**Files expected in `/content/` (Colab root):** `train.csv`, `test.csv`, `sample_submission.csv`


## 1. 📦 Install & Imports

In [ ]:
# Install extra libraries
!pip install lightgbm xgboost catboost optuna -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, re
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style='whitegrid', palette='husl')
SEED = 42
np.random.seed(SEED)
print("All imports successful ✅")


## 2. 📂 Load Data

In [ ]:
train = pd.read_csv('/content/train.csv')
test  = pd.read_csv('/content/test.csv')
sub   = pd.read_csv('/content/sample_submission.csv')

print(f"Train : {train.shape}   Test : {test.shape}")
train.head(3)


## 3. 🔍 Exploratory Data Analysis

In [ ]:
# ── Basic info ──────────────────────────────────
print("=== TRAIN INFO ===")
train.info()
print("\n=== MISSING VALUES ===")
print(train.isnull().sum())
print("\n=== STATISTICS ===")
train.describe()


In [ ]:
# ── Target distribution ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(train['demand'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Demand Distribution')
axes[0].set_xlabel('demand')

axes[1].hist(np.log1p(train['demand']), bins=60, color='salmon', edgecolor='white')
axes[1].set_title('log1p(Demand) Distribution')
axes[1].set_xlabel('log1p(demand)')

axes[2].boxplot(train['demand'])
axes[2].set_title('Demand Box-Plot')

plt.tight_layout()
plt.show()
print(f"Skewness : {train['demand'].skew():.3f}  |  Kurtosis : {train['demand'].kurt():.3f}")


In [ ]:
# ── Categorical columns ─────────────────────────
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, col in enumerate(cat_cols):
    counts = train[col].value_counts(dropna=False)
    axes[0, i].bar(counts.index.astype(str), counts.values, color='teal')
    axes[0, i].set_title(f'{col} — count')
    axes[0, i].tick_params(axis='x', rotation=30)
    
    means = train.groupby(col, dropna=False)['demand'].mean()
    axes[1, i].bar(means.index.astype(str), means.values, color='coral')
    axes[1, i].set_title(f'{col} — mean demand')
    axes[1, i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# ── Timestamp & day patterns ────────────────────
# Parse timestamp → hour + minute → fractional hour
def parse_ts(ts):
    h, m = ts.split(':')
    return int(h) + int(m) / 60

train['hour'] = train['timestamp'].apply(parse_ts)
test['hour']  = test['timestamp'].apply(parse_ts)

hourly_demand = train.groupby('hour')['demand'].mean()

plt.figure(figsize=(14, 4))
plt.plot(hourly_demand.index, hourly_demand.values, marker='o', linewidth=2, color='royalblue')
plt.fill_between(hourly_demand.index, hourly_demand.values, alpha=0.2)
plt.title('Average Demand by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Mean Demand')
plt.xticks(range(0, 25, 1))
plt.tight_layout()
plt.show()


In [ ]:
# ── Geohash analysis ───────────────────────────
# Prefix captures spatial resolution levels
for length in [3, 4, 5]:
    train[f'geo{length}'] = train['geohash'].str[:length]

top_geo = train['geohash'].value_counts().head(20)
plt.figure(figsize=(14, 4))
plt.bar(top_geo.index, top_geo.values, color='mediumseagreen')
plt.title('Top 20 Geohash Locations by Count')
plt.xlabel('geohash')
plt.ylabel('count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Mean demand per geohash prefix (geo4)
geo4_demand = train.groupby('geo4')['demand'].mean().sort_values(ascending=False).head(20)
plt.figure(figsize=(14, 4))
plt.bar(geo4_demand.index, geo4_demand.values, color='mediumpurple')
plt.title('Top 20 geo4 Prefixes — Mean Demand')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ── Correlation heatmap ─────────────────────────
num_cols = ['demand', 'hour', 'day', 'NumberofLanes', 'Temperature']
plt.figure(figsize=(8, 6))
sns.heatmap(train[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()


In [ ]:
# ── Temperature vs Demand ───────────────────────
plt.figure(figsize=(8, 5))
plt.scatter(train['Temperature'], train['demand'], alpha=0.05, s=5, color='steelblue')
plt.xlabel('Temperature')
plt.ylabel('Demand')
plt.title('Temperature vs Demand')
plt.tight_layout()
plt.show()


## 4. 🛠️ Feature Engineering

In [ ]:
def feature_engineering(df, is_train=True):
    df = df.copy()
    
    # ── Timestamp features ──────────────────────
    def parse_ts(ts):
        h, m = ts.split(':')
        return int(h), int(m)
    
    df['hour']   = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15          # 96 daily slots
    
    # Cyclical encoding of hour (captures 0≈23 proximity)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    
    # Peak periods
    df['is_morning_peak'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
    df['is_evening_peak'] = ((df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)
    df['is_night']        = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['is_midday']       = ((df['hour'] >= 11) & (df['hour'] <= 13)).astype(int)
    
    # ── Geohash features ───────────────────────
    for length in [3, 4, 5, 6]:
        df[f'geo{length}'] = df['geohash'].str[:length]
    
    # ── Encode categoricals ────────────────────
    binary_map = {'Yes': 1, 'No': 0, 'Allowed': 1, 'Not Allowed': 0}
    df['LargeVehicles_enc'] = df['LargeVehicles'].map(binary_map)
    df['Landmarks_enc']     = df['Landmarks'].map(binary_map)
    
    road_map = {'Highway': 3, 'Street': 2, 'Residential': 1}
    df['RoadType_enc'] = df['RoadType'].map(road_map)           # NaN → missing
    
    weather_map = {'Sunny': 0, 'Foggy': 1, 'Rainy': 2, 'Snowy': 3}
    df['Weather_enc'] = df['Weather'].map(weather_map)
    
    # ── Interaction features ───────────────────
    df['lane_large'] = df['NumberofLanes'] * df['LargeVehicles_enc']
    df['lane_road']  = df['NumberofLanes'] * df['RoadType_enc'].fillna(0)
    
    # ── Temperature features ───────────────────
    df['temp_missing'] = df['Temperature'].isna().astype(int)
    # Interaction: extreme cold/heat → lower demand perhaps
    df['temp_sq'] = df['Temperature'].fillna(df['Temperature'].median()) ** 2
    
    return df

train_fe = feature_engineering(train, is_train=True)
test_fe  = feature_engineering(test,  is_train=False)

print("Feature engineering done ✅")
print("New shape — train:", train_fe.shape, "  test:", test_fe.shape)


In [ ]:
# ── Target-encode high-cardinality geohash cols ─
# Use train mean demand per geohash, geo4, geo5
from sklearn.model_selection import KFold

def target_encode_kfold(train_df, test_df, col, target='demand', n_splits=5, smoothing=10):
    """KFold target encoding to avoid leakage."""
    global_mean = train_df[target].mean()
    # For test: use full-train statistics
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['smooth'] = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
    
    # Train: out-of-fold
    oof = np.zeros(len(train_df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr_idx, val_idx in kf.split(train_df):
        fold_stats = train_df.iloc[tr_idx].groupby(col)[target].agg(['mean', 'count'])
        fold_stats['smooth'] = (fold_stats['mean'] * fold_stats['count'] + global_mean * smoothing) / (fold_stats['count'] + smoothing)
        oof[val_idx] = train_df.iloc[val_idx][col].map(fold_stats['smooth']).fillna(global_mean)
    
    test_enc = test_df[col].map(stats['smooth']).fillna(global_mean)
    return oof, test_enc.values

for geo_col in ['geohash', 'geo3', 'geo4', 'geo5']:
    tr_enc, te_enc = target_encode_kfold(train_fe, test_fe, geo_col)
    train_fe[f'{geo_col}_te'] = tr_enc
    test_fe[f'{geo_col}_te']  = te_enc

print("Target encoding done ✅")


In [ ]:
# ── Also encode geohash × time interaction ───────
# Mean demand per (geohash, time_slot) — very predictive
combo_mean = train_fe.groupby(['geohash', 'time_slot'])['demand'].mean()
train_fe['geo_slot_te'] = train_fe.set_index(['geohash', 'time_slot']).index.map(combo_mean)
test_fe['geo_slot_te']  = test_fe.set_index(['geohash', 'time_slot']).index.map(combo_mean)

global_mean = train_fe['demand'].mean()
train_fe['geo_slot_te'].fillna(global_mean, inplace=True)
test_fe['geo_slot_te'].fillna(global_mean, inplace=True)

print("geo×slot interaction TE done ✅")


## 5. 🧮 Prepare Feature Matrix

In [ ]:
FEATURE_COLS = [
    # Time
    'hour', 'minute', 'time_slot', 'day',
    'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
    'is_morning_peak', 'is_evening_peak', 'is_night', 'is_midday',
    # Road
    'NumberofLanes', 'RoadType_enc', 'LargeVehicles_enc',
    'lane_large', 'lane_road',
    # Location
    'Landmarks_enc',
    'geohash_te', 'geo3_te', 'geo4_te', 'geo5_te', 'geo_slot_te',
    # Weather & Temp
    'Weather_enc', 'Temperature', 'temp_sq', 'temp_missing',
]

TARGET = 'demand'

X = train_fe[FEATURE_COLS].copy()
y = train_fe[TARGET].values
X_test = test_fe[FEATURE_COLS].copy()

# Impute remaining NaNs (Temperature, RoadType_enc, Weather_enc)
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='median')
X      = pd.DataFrame(imp.fit_transform(X),      columns=FEATURE_COLS)
X_test = pd.DataFrame(imp.transform(X_test),     columns=FEATURE_COLS)

print(f"X shape: {X.shape}   X_test shape: {X_test.shape}")
print(f"Any NaN in X? {X.isnull().any().any()}   X_test? {X_test.isnull().any().any()}")


## 6. 📊 Baseline Models
5-Fold CV — tracking R² and OOF predictions

In [ ]:
def evaluate_kfold(model, X, y, n_splits=5, model_name='Model'):
    """Returns OOF predictions, OOF R², list of fold scores."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_preds = np.zeros(len(y))
    scores = []
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx],      y[val_idx]
        
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        preds = np.clip(preds, 0, 1)
        
        fold_score = max(0, 100 * r2_score(y_val, preds))
        scores.append(fold_score)
        oof_preds[val_idx] = preds
        print(f"  Fold {fold+1}: {fold_score:.4f}")
    
    oof_score = max(0, 100 * r2_score(y, oof_preds))
    print(f"  {model_name} → OOF R²-score: {oof_score:.4f} | CV mean: {np.mean(scores):.4f} ± {np.std(scores):.4f}\n")
    return oof_preds, oof_score, scores

results = {}  # will store {name: (oof_preds, oof_score)}


In [ ]:
# ── Ridge Regression ────────────────────────────
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ridge = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))])
print("Ridge Regression:")
oof, score, _ = evaluate_kfold(ridge, X, y, model_name='Ridge')
results['Ridge'] = (oof, score)


In [ ]:
# ── Extra Trees ─────────────────────────────────
from sklearn.ensemble import ExtraTreesRegressor
et = ExtraTreesRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
print("Extra Trees:")
oof, score, _ = evaluate_kfold(et, X, y, model_name='ExtraTrees')
results['ExtraTrees'] = (oof, score)


In [ ]:
# ── LightGBM ────────────────────────────────────
lgb_params = dict(
    n_estimators=1000, learning_rate=0.05,
    num_leaves=127, max_depth=-1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    min_child_samples=20,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm = lgb.LGBMRegressor(**lgb_params)
print("LightGBM:")
oof_lgb, score_lgb, _ = evaluate_kfold(lgbm, X, y, model_name='LightGBM')
results['LightGBM'] = (oof_lgb, score_lgb)


In [ ]:
# ── XGBoost ─────────────────────────────────────
xgb_params = dict(
    n_estimators=1000, learning_rate=0.05,
    max_depth=7, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, tree_method='hist'
)
xgbm = xgb.XGBRegressor(**xgb_params)
print("XGBoost:")
oof_xgb, score_xgb, _ = evaluate_kfold(xgbm, X, y, model_name='XGBoost')
results['XGBoost'] = (oof_xgb, score_xgb)


In [ ]:
# ── CatBoost ────────────────────────────────────
cat = CatBoostRegressor(
    iterations=1000, learning_rate=0.05,
    depth=8, l2_leaf_reg=3, random_seed=SEED, verbose=0
)
print("CatBoost:")
oof_cat, score_cat, _ = evaluate_kfold(cat, X, y, model_name='CatBoost')
results['CatBoost'] = (oof_cat, score_cat)


In [ ]:
# ── Baseline Summary ────────────────────────────
print("\n=== BASELINE LEADERBOARD ===")
baseline_df = pd.DataFrame(
    [(k, v[1]) for k, v in results.items()],
    columns=['Model', 'OOF_Score']
).sort_values('OOF_Score', ascending=False)
print(baseline_df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.barh(baseline_df['Model'], baseline_df['OOF_Score'], color='steelblue')
plt.xlabel('OOF Score (max 100)')
plt.title('Baseline Model Comparison')
plt.tight_layout()
plt.show()


## 7. 🔧 Hyperparameter Tuning with Optuna (LightGBM)
(Best baseline from previous section)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 500, 2000),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 31, 255),
        max_depth         = trial.suggest_int('max_depth', 4, 12),
        min_child_samples = trial.suggest_int('min_child_samples', 10, 50),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    model = lgb.LGBMRegressor(**params)
    kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, val_idx in kf.split(X):
        model.fit(X.iloc[tr_idx], y[tr_idx])
        preds = np.clip(model.predict(X.iloc[val_idx]), 0, 1)
        scores.append(r2_score(y[val_idx], preds))
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best score:", study.best_value)
print("Best params:", study.best_params)


In [ ]:
# ── Retrain best LGBM on full data & OOF ────────
best_lgb = lgb.LGBMRegressor(**study.best_params, random_state=SEED, n_jobs=-1, verbose=-1)
print("Tuned LightGBM:")
oof_lgb_tuned, score_lgb_tuned, _ = evaluate_kfold(best_lgb, X, y, model_name='LightGBM_Tuned')
results['LightGBM_Tuned'] = (oof_lgb_tuned, score_lgb_tuned)


## 8. 🧠 Deep Learning — Multi-Layer Perceptron (PyTorch)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

class TrafficMLP(nn.Module):
    def __init__(self, in_dim, hidden=[512, 256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers += [nn.Linear(prev, 1), nn.Sigmoid()]   # Sigmoid → clamp to [0,1]
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp(X_tr_arr, y_tr_arr, X_val_arr, y_val_arr,
              epochs=60, batch_size=2048, lr=1e-3):
    
    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr_arr)
    X_val_sc = scaler.transform(X_val_arr)
    
    tr_ds  = TensorDataset(torch.tensor(X_tr_sc,  dtype=torch.float32).to(device),
                           torch.tensor(y_tr_arr, dtype=torch.float32).to(device))
    val_ds = TensorDataset(torch.tensor(X_val_sc, dtype=torch.float32).to(device),
                           torch.tensor(y_val_arr,dtype=torch.float32).to(device))
    
    tr_loader  = DataLoader(tr_ds,  batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    
    model = TrafficMLP(in_dim=X_tr_arr.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    best_state    = None
    
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        model.eval()
        with torch.no_grad():
            val_preds = torch.cat([model(xb) for xb, _ in val_loader]).cpu().numpy()
        val_loss = np.mean((val_preds - y_val_arr) ** 2)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        if epoch % 10 == 0:
            r2 = r2_score(y_val_arr, np.clip(val_preds, 0, 1))
            print(f"  Epoch {epoch:3d} | val_loss: {val_loss:.6f} | R²: {r2:.4f}")
    
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_preds = torch.cat([model(xb) for xb, _ in val_loader]).cpu().numpy()
    return model, scaler, np.clip(val_preds, 0, 1)


# ── KFold MLP ───────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_mlp = np.zeros(len(y))
mlp_models = []    # store (model, scaler) for test prediction

print("Training MLP (5-Fold):")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n  == Fold {fold+1} ==")
    X_tr_arr  = X.values[tr_idx].astype(np.float32)
    X_val_arr = X.values[val_idx].astype(np.float32)
    y_tr_arr  = y[tr_idx].astype(np.float32)
    y_val_arr = y[val_idx].astype(np.float32)
    
    model, scaler, val_preds = train_mlp(X_tr_arr, y_tr_arr, X_val_arr, y_val_arr)
    oof_mlp[val_idx] = val_preds
    mlp_models.append((model, scaler))

mlp_score = max(0, 100 * r2_score(y, oof_mlp))
print(f"\nMLP OOF Score: {mlp_score:.4f}")
results['MLP'] = (oof_mlp, mlp_score)


## 9. 🧠 Deep Learning — Residual MLP (Skip Connections)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResidualMLP(nn.Module):
    def __init__(self, in_dim, dim=256, n_blocks=4, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(in_dim, dim), nn.BatchNorm1d(dim), nn.ReLU())
        self.blocks = nn.Sequential(*[ResBlock(dim, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x):
        return self.head(self.blocks(self.input_proj(x))).squeeze(1)


def train_res_mlp(X_tr_arr, y_tr_arr, X_val_arr, y_val_arr,
                  epochs=60, batch_size=2048, lr=1e-3):
    scaler = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr_arr)
    X_val_sc = scaler.transform(X_val_arr)
    
    tr_ds  = TensorDataset(torch.tensor(X_tr_sc,  dtype=torch.float32).to(device),
                           torch.tensor(y_tr_arr, dtype=torch.float32).to(device))
    val_ds = TensorDataset(torch.tensor(X_val_sc, dtype=torch.float32).to(device),
                           torch.tensor(y_val_arr,dtype=torch.float32).to(device))
    
    tr_loader  = DataLoader(tr_ds,  batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    
    model = ResidualMLP(in_dim=X_tr_arr.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    best_state    = None
    
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        model.eval()
        with torch.no_grad():
            val_preds = torch.cat([model(xb) for xb, _ in val_loader]).cpu().numpy()
        val_loss = np.mean((val_preds - y_val_arr) ** 2)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        if epoch % 10 == 0:
            r2 = r2_score(y_val_arr, np.clip(val_preds, 0, 1))
            print(f"  Epoch {epoch:3d} | val_loss: {val_loss:.6f} | R²: {r2:.4f}")
    
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_preds = torch.cat([model(xb) for xb, _ in val_loader]).cpu().numpy()
    return model, scaler, np.clip(val_preds, 0, 1)


kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_res = np.zeros(len(y))
res_models = []

print("Training ResidualMLP (5-Fold):")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n  == Fold {fold+1} ==")
    model, scaler, val_preds = train_res_mlp(
        X.values[tr_idx].astype(np.float32), y[tr_idx].astype(np.float32),
        X.values[val_idx].astype(np.float32), y[val_idx].astype(np.float32)
    )
    oof_res[val_idx] = val_preds
    res_models.append((model, scaler))

res_score = max(0, 100 * r2_score(y, oof_res))
print(f"\nResidualMLP OOF Score: {res_score:.4f}")
results['ResidualMLP'] = (oof_res, res_score)


## 10. 🔗 Fusion & Stacking Ensemble

In [ ]:
# ── Show all OOF scores ─────────────────────────
print("\n=== ALL MODELS OOF SCORES ===")
for name, (oof, score) in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:20s} : {score:.4f}")


In [ ]:
# ── 10A: Weighted Average (manual) ──────────────
# Weight = softmax of OOF scores (higher score → higher weight)
scores_arr = np.array([results[k][1] for k in results])
weights    = np.exp(scores_arr - scores_arr.max())
weights    /= weights.sum()

model_names = list(results.keys())
oof_stack   = np.column_stack([results[k][0] for k in model_names])

oof_weighted = (oof_stack * weights).sum(axis=1)
w_score = max(0, 100 * r2_score(y, np.clip(oof_weighted, 0, 1)))
print(f"Weighted Average OOF Score: {w_score:.4f}")

for name, w in zip(model_names, weights):
    print(f"  {name:20s} weight: {w:.4f}")


In [ ]:
# ── 10B: Meta-learner (Stacking) ────────────────
# Level-1 features = OOF predictions of all base models
# Meta-learner = Ridge + LightGBM blend

from sklearn.linear_model import Ridge

meta_X = oof_stack.copy()
meta_y = y.copy()

# Ridge meta
ridge_meta = Ridge(alpha=1.0)
kf_meta = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_meta_ridge = np.zeros(len(y))
for tr_idx, val_idx in kf_meta.split(meta_X):
    ridge_meta.fit(meta_X[tr_idx], meta_y[tr_idx])
    oof_meta_ridge[val_idx] = ridge_meta.predict(meta_X[val_idx])

ridge_meta_score = max(0, 100 * r2_score(y, np.clip(oof_meta_ridge, 0, 1)))
print(f"Ridge Meta-learner OOF Score: {ridge_meta_score:.4f}")

# LightGBM meta
lgb_meta = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05,
                              num_leaves=31, random_state=SEED, verbose=-1)
oof_meta_lgb = np.zeros(len(y))
for tr_idx, val_idx in kf_meta.split(meta_X):
    lgb_meta.fit(meta_X[tr_idx], meta_y[tr_idx])
    oof_meta_lgb[val_idx] = lgb_meta.predict(meta_X[val_idx])

lgb_meta_score = max(0, 100 * r2_score(y, np.clip(oof_meta_lgb, 0, 1)))
print(f"LGB  Meta-learner OOF Score: {lgb_meta_score:.4f}")

# Blend meta predictions
oof_final_stack = 0.5 * oof_meta_ridge + 0.5 * oof_meta_lgb
stack_score = max(0, 100 * r2_score(y, np.clip(oof_final_stack, 0, 1)))
print(f"Blended  Meta-learner OOF Score: {stack_score:.4f}")


In [ ]:
# ── Final ensemble score comparison ─────────────
results['WeightedAvg'] = (np.clip(oof_weighted, 0, 1), w_score)
results['StackRidge']  = (np.clip(oof_meta_ridge, 0, 1), ridge_meta_score)
results['StackLGB']    = (np.clip(oof_meta_lgb, 0, 1), lgb_meta_score)
results['StackBlend']  = (np.clip(oof_final_stack, 0, 1), stack_score)

all_results_df = pd.DataFrame(
    [(k, v[1]) for k, v in results.items()],
    columns=['Model', 'OOF_Score']
).sort_values('OOF_Score', ascending=False)

print(all_results_df.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(all_results_df['Model'][::-1], all_results_df['OOF_Score'][::-1],
         color='steelblue', edgecolor='white')
plt.xlabel('OOF Score (max 100)')
plt.title('All Models — OOF Score Leaderboard')
plt.tight_layout()
plt.show()


## 11. 📤 Generate Test Predictions & Submit

In [ ]:
# ── Retrain all base models on full train ────────
# Choose best ensemble strategy based on above

# 1. LightGBM (tuned) — full train
best_lgb.fit(X, y)
test_lgb = np.clip(best_lgb.predict(X_test), 0, 1)

# 2. CatBoost — full train
cat.fit(X, y, verbose=0)
test_cat = np.clip(cat.predict(X_test), 0, 1)

# 3. XGBoost — full train
xgbm.fit(X, y)
test_xgb = np.clip(xgbm.predict(X_test), 0, 1)

# 4. MLP — average all fold models
X_test_np = X_test.values.astype(np.float32)
test_mlp_preds = []
for model, scaler in mlp_models:
    X_sc = scaler.transform(X_test_np)
    X_t  = torch.tensor(X_sc, dtype=torch.float32).to(device)
    with torch.no_grad():
        preds = model(X_t).cpu().numpy()
    test_mlp_preds.append(np.clip(preds, 0, 1))
test_mlp = np.mean(test_mlp_preds, axis=0)

# 5. ResidualMLP — average all fold models
test_res_preds = []
for model, scaler in res_models:
    X_sc = scaler.transform(X_test_np)
    X_t  = torch.tensor(X_sc, dtype=torch.float32).to(device)
    with torch.no_grad():
        preds = model(X_t).cpu().numpy()
    test_res_preds.append(np.clip(preds, 0, 1))
test_res = np.mean(test_res_preds, axis=0)

print("All test predictions generated ✅")


In [ ]:
# ── Blend test predictions ───────────────────────
# Use same weights as best ensemble (adjust based on OOF scores above)

# Simple equal-weight fusion of top models
test_ensemble = (
    0.30 * test_lgb +
    0.25 * test_cat +
    0.20 * test_xgb +
    0.15 * test_mlp +
    0.10 * test_res
)
test_ensemble = np.clip(test_ensemble, 0, 1)

print(f"Test ensemble stats — min: {test_ensemble.min():.4f}  max: {test_ensemble.max():.4f}  mean: {test_ensemble.mean():.4f}")

# Distribution check
plt.figure(figsize=(8, 4))
plt.hist(test_ensemble, bins=60, color='steelblue', edgecolor='white')
plt.title('Test Prediction Distribution')
plt.xlabel('Predicted Demand')
plt.tight_layout()
plt.show()


In [ ]:
# ── Save submission ──────────────────────────────
submission = pd.DataFrame({
    'Index' : test['Index'].values,
    'demand': test_ensemble
})

assert submission.shape == (41778, 2), f"Wrong shape: {submission.shape}"
assert not submission.isnull().any().any(), "NaN in submission!"

submission.to_csv('/content/submission.csv', index=False)
print("submission.csv saved ✅")
print(submission.head(10))


## 12. 📈 Feature Importance

In [ ]:
# Feature importance from best LGBM
best_lgb.fit(X, y)
fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': best_lgb.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(fi['feature'][::-1], fi['importance'][::-1], color='steelblue')
plt.title('LightGBM Feature Importance')
plt.tight_layout()
plt.show()


## 13. 💡 Tips for Further Improvement

### Feature Engineering
- **Lag features**: mean demand for same (geohash, time_slot) from previous days
- **Rolling stats**: 3h / 6h rolling mean per geohash
- **Geohash decoding**: decode to lat/lon, compute distance to city center

### Modelling
- **TabNet** (`pip install pytorch-tabnet`) — attention-based tabular DL
- **FT-Transformer** — transformer architecture for tabular data
- **LSTM/GRU** — if you reshape data as time-series per geohash
- **More Optuna trials** — increase `n_trials=200` for LGBM/XGB

### Ensemble
- **Diverse models** → ExtraTrees + GBM + NN blend usually beats any single model
- **Layer-2 stacking**: add OOF as feature + original features to meta-learner
- **Rank averaging**: blend raw predictions + rank-transformed predictions
